<a href="https://colab.research.google.com/github/kartik815/Amazon-ML-Challenge-2026/blob/main/notebooks/07_Candidate_Scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install rapidfuzz indic-transliteration scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.9/162.9 kB 12.0 MB/s eta 0:00:00


In [2]:
import os
import gc
import json
import re
import math
import warnings

import numpy as np
import pandas as pd

from rapidfuzz import fuzz
from indic_transliteration import sanscript

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    fbeta_score
)

import joblib

warnings.filterwarnings("ignore")

print("Libraries loaded.")

Libraries loaded.


In [12]:
import os

DRIVE_ROOT = "/content/gdrive/MyDrive/Amazon ML Challenge 2026"

CLEANED_DATA_ROOT = os.path.join(
    DRIVE_ROOT,
    "03_Experiments",
    "Cleaned_Data"
)

TRAIN_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset",
    "train"
)

CANDIDATE_ROOT = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs"
)

MODEL_ROOT = os.path.join(
    DRIVE_ROOT,
    "04_Models"
)

SCORING_ROOT = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Scored_Candidates"
)

os.makedirs(
    SCORING_ROOT,
    exist_ok=True
)

S1_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s1_cleaned.tsv"
)

S2_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s2_cleaned.tsv"
)

S3_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s3_cleaned.tsv"
)

GT_PATH = os.path.join(
    TRAIN_ROOT,
    "train_ground_truth.tsv"
)

CANDIDATE_PATH = os.path.join(
    CANDIDATE_ROOT,
    "candidate_pairs.tsv"
)

MODEL_PATH = os.path.join(
    MODEL_ROOT,
    "entity_resolution_logistic_model.joblib"
)

TRAIN_PREDICTIONS_PATH = os.path.join(
    SCORING_ROOT,
    "train_candidate_matches.tsv"
)

TRAIN_STATS_PATH = os.path.join(
    SCORING_ROOT,
    "train_scoring_stats.json"
)

print("Paths configured.")
print()
print("S1:", S1_PATH)
print("S2:", S2_PATH)
print("S3:", S3_PATH)
print("GT:", GT_PATH)
print("Candidates:", CANDIDATE_PATH)
print("Model:", MODEL_PATH)

Paths configured.

S1: /content/gdrive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data/train_s1_cleaned.tsv
S2: /content/gdrive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data/train_s2_cleaned.tsv
S3: /content/gdrive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data/train_s3_cleaned.tsv
GT: /content/gdrive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/train/train_ground_truth.tsv
Candidates: /content/gdrive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs/candidate_pairs.tsv
Model: /content/gdrive/MyDrive/Amazon ML Challenge 2026/04_Models/entity_resolution_logistic_model.joblib


In [7]:
from google.colab import drive
import os

MOUNT_POINT = "/content/gdrive"

# Mount Google Drive at a clean location
drive.mount(MOUNT_POINT)

print("Drive mounted:", os.path.exists(MOUNT_POINT))

Mounted at /content/gdrive
Drive mounted: True


In [8]:
DRIVE_ROOT = "/content/gdrive/MyDrive/Amazon ML Challenge 2026"

print("Project root exists:", os.path.exists(DRIVE_ROOT))

if os.path.exists(DRIVE_ROOT):
    print("\nProject folders:")
    print(os.listdir(DRIVE_ROOT))

Project root exists: True

Project folders:
['01_Dataset', '05_Outputs', '02_Notebooks', '03_Experiments', '04_Models', '06_Documentation']


In [9]:
matches = []

for root, dirs, files in os.walk(DRIVE_ROOT):
    if "candidate_pairs.tsv" in files:
        matches.append(
            os.path.join(root, "candidate_pairs.tsv")
        )

print("Candidate files found:")

for path in matches:
    print(path)

Candidate files found:
/content/gdrive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs/candidate_pairs.tsv


In [10]:
if not matches:
    raise FileNotFoundError(
        "candidate_pairs.tsv was not found in the project folder."
    )

CANDIDATE_PATH = matches[0]

print("Using:")
print(CANDIDATE_PATH)

size_gb = os.path.getsize(CANDIDATE_PATH) / (1024**3)

print(f"Size: {size_gb:.2f} GB")

Using:
/content/gdrive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs/candidate_pairs.tsv
Size: 2.00 GB


In [11]:
candidate_preview = pd.read_csv(
    CANDIDATE_PATH,
    sep="\t",
    nrows=5
)

print(candidate_preview.columns.tolist())

display(candidate_preview)

['source1_entity_id', 'candidate_entity_id', 'source']


,source1_entity_id,candidate_entity_id,source
0,S1-925783039,S2-157377754,S2
1,S1-925783039,S3-997698194,S3
2,S1-773889195,S2-374107005,S2
3,S1-773889195,S3-158745057,S3
4,S1-773889195,S3-189140669,S3


In [13]:
required_files = {
    "S1": S1_PATH,
    "S2": S2_PATH,
    "S3": S3_PATH,
    "Ground Truth": GT_PATH,
    "Candidate Pairs": CANDIDATE_PATH
}

for name, path in required_files.items():
    print(
        f"{name:15} : "
        f"{'OK' if os.path.exists(path) else 'MISSING'}"
    )

S1              : OK
S2              : OK
S3              : OK
Ground Truth    : OK
Candidate Pairs : OK


In [14]:
print(
    "Existing model:",
    os.path.exists(MODEL_PATH)
)

if os.path.exists(MODEL_PATH):
    print(
        f"Model size: "
        f"{os.path.getsize(MODEL_PATH) / 1024:.1f} KB"
    )

Existing model: False


In [15]:
gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype="string"
)

gt["matched_entity_ids"] = (
    gt["matched_entity_ids"]
    .fillna("")
    .astype("string")
)

gt_map = gt.set_index(
    "source1_entity_id"
)["matched_entity_ids"]

print("Ground-truth rows:", len(gt))

Ground-truth rows: 2206821


In [16]:
SOURCE_COLUMNS = [
    "entity_id",
    "country_norm",
    "name_core",
    "address_norm"
]

s1_lookup = pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=SOURCE_COLUMNS,
    dtype="string"
).set_index("entity_id")

print("S1 shape:", s1_lookup.shape)

S1 shape: (2206821, 3)


In [17]:
NAME_STOPWORDS = {
    "inc", "incorporated", "llc", "ltd", "limited",
    "pvt", "private", "company", "co", "corp",
    "corporation", "group", "services", "partners",
    "holdings", "associates", "center", "india",
    "and", "of"
}

FEATURE_COLUMNS = [
    "name_char_similarity",
    "name_3gram_jaccard",
    "name_token_jaccard",
    "address_char_similarity",
    "address_token_jaccard",
    "country_exact",
    "translit_name_char_similarity"
]

In [18]:
from rapidfuzz import fuzz
from indic_transliteration import sanscript

def safe_text(value):
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value).strip().lower()


def meaningful_tokens(text):
    text = safe_text(text)

    if not text:
        return []

    return [
        token
        for token in text.split()
        if token not in NAME_STOPWORDS
        and len(token) >= 2
    ]


def token_jaccard(a, b):
    a = set(meaningful_tokens(a))
    b = set(meaningful_tokens(b))

    if not a and not b:
        return 1.0

    if not a or not b:
        return 0.0

    return len(a & b) / len(a | b)


def char_ngram_jaccard(a, b, n=3):
    a = safe_text(a)
    b = safe_text(b)

    if len(a) < n or len(b) < n:
        return 0.0

    ga = {
        a[i:i+n]
        for i in range(len(a)-n+1)
    }

    gb = {
        b[i:i+n]
        for i in range(len(b)-n+1)
    }

    if not ga and not gb:
        return 1.0

    if not ga or not gb:
        return 0.0

    return len(ga & gb) / len(ga | gb)


def transliterate_name(text):
    text = safe_text(text)

    if not text:
        return ""

    # Keep ordinary Latin names unchanged
    if all(ord(c) < 128 for c in text):
        return text

    try:
        return sanscript.transliterate(
            text,
            sanscript.AUTO,
            sanscript.IAST
        ).lower()
    except Exception:
        return text


def pair_features(
    name1,
    address1,
    country1,
    name2,
    address2,
    country2
):

    n1 = safe_text(name1)
    n2 = safe_text(name2)

    a1 = safe_text(address1)
    a2 = safe_text(address2)

    c1 = safe_text(country1)
    c2 = safe_text(country2)

    t1 = transliterate_name(n1)
    t2 = transliterate_name(n2)

    return {
        "name_char_similarity":
            fuzz.ratio(n1, n2) / 100.0,

        "name_3gram_jaccard":
            char_ngram_jaccard(n1, n2, 3),

        "name_token_jaccard":
            token_jaccard(n1, n2),

        "address_char_similarity":
            fuzz.ratio(a1, a2) / 100.0
            if a1 and a2 else 0.0,

        "address_token_jaccard":
            token_jaccard(a1, a2)
            if a1 and a2 else 0.0,

        "country_exact":
            1.0 if c1 and c1 == c2 else 0.0,

        "translit_name_char_similarity":
            fuzz.ratio(t1, t2) / 100.0
    }

In [19]:
import gc
import numpy as np
import pandas as pd

RANDOM_SEED = 42

POSITIVE_TARGET = 20_000
NEGATIVE_TARGET = 20_000

CANDIDATE_CHUNK = 500_000

rng = np.random.default_rng(RANDOM_SEED)

positive_samples = []
negative_samples = []

positive_count = 0
negative_count = 0

chunk_number = 0

reader = pd.read_csv(
    CANDIDATE_PATH,
    sep="\t",
    usecols=[
        "source1_entity_id",
        "candidate_entity_id",
        "source"
    ],
    dtype="string",
    chunksize=CANDIDATE_CHUNK
)

for chunk in reader:

    chunk_number += 1

    # --------------------------------------------------------
    # Determine whether each candidate is a true match
    # --------------------------------------------------------

    for source_name in ["S2", "S3"]:

        source_chunk = chunk[
            chunk["source"] == source_name
        ].copy()

        if source_chunk.empty:
            continue

        # Ground truth for these S1 entities
        gt_strings = gt_map.reindex(
            source_chunk["source1_entity_id"]
        ).fillna("")

        # Build true-pair mask
        true_mask = np.zeros(
            len(source_chunk),
            dtype=bool
        )

        for i, (
            s1_id,
            candidate_id
        ) in enumerate(
            zip(
                source_chunk["source1_entity_id"],
                source_chunk["candidate_entity_id"]
            )
        ):

            matched_string = gt_strings.iloc[i]

            if not matched_string:
                continue

            # Ground truth IDs are comma-separated
            true_ids = matched_string.split(",")

            true_mask[i] = (
                str(candidate_id).strip()
                in {
                    x.strip()
                    for x in true_ids
                    if x.strip()
                }
            )

        source_chunk["label"] = (
            true_mask.astype(np.int8)
        )

        # ----------------------------------------------------
        # Collect positives
        # ----------------------------------------------------

        positives = source_chunk[
            source_chunk["label"] == 1
        ]

        if len(positives) > 0:

            remaining = (
                POSITIVE_TARGET
                - positive_count
            )

            if remaining > 0:

                take = min(
                    remaining,
                    len(positives)
                )

                selected = positives.sample(
                    n=take,
                    random_state=RANDOM_SEED + chunk_number
                )

                positive_samples.append(
                    selected
                )

                positive_count += take

        # ----------------------------------------------------
        # Collect negatives
        # ----------------------------------------------------

        negatives = source_chunk[
            source_chunk["label"] == 0
        ]

        if len(negatives) > 0:

            remaining = (
                NEGATIVE_TARGET
                - negative_count
            )

            if remaining > 0:

                take = min(
                    remaining,
                    len(negatives)
                )

                selected = negatives.sample(
                    n=take,
                    random_state=RANDOM_SEED + chunk_number + 10000
                )

                negative_samples.append(
                    selected
                )

                negative_count += take

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if chunk_number % 10 == 0:

        print(
            f"Chunks: {chunk_number:,} | "
            f"Positives: {positive_count:,} | "
            f"Negatives: {negative_count:,}"
        )

    # Stop as soon as we have enough
    if (
        positive_count >= POSITIVE_TARGET
        and
        negative_count >= NEGATIVE_TARGET
    ):
        print(
            "\nEnough training examples collected."
        )
        break

    del chunk
    gc.collect()


training_pairs = pd.concat(
    positive_samples + negative_samples,
    ignore_index=True
)

training_pairs = training_pairs.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

print("\nTraining dataset:")
print(training_pairs.shape)

print("\nClass distribution:")
print(
    training_pairs["label"]
    .value_counts()
)

print("\nSource distribution:")
print(
    training_pairs["source"]
    .value_counts()
)

display(
    training_pairs.head(10)
)


Enough training examples collected.

Training dataset:
(40000, 4)

Class distribution:
label
0    20000
1    20000
Name: count, dtype: int64

Source distribution:
source
S2    30741
S3     9259
Name: count, dtype: Int64


,source1_entity_id,candidate_entity_id,source,label
0,S1-108314429,S2-779676303,S2,0
1,S1-479783161,S3-721023966,S3,1
2,S1-742646574,S2-858112042,S2,0
3,S1-810841075,S2-876563385,S2,1
4,S1-305618376,S2-25232162,S2,0
5,S1-821827055,S2-261347134,S2,0
6,S1-502275537,S3-771295196,S3,1
7,S1-899990335,S2-213758143,S2,0
8,S1-719305733,S3-262389914,S3,1
9,S1-824373227,S2-392253580,S2,0


In [20]:
def load_source_lookup(path):
    return pd.read_csv(
        path,
        sep="\t",
        usecols=[
            "entity_id",
            "country_norm",
            "name_core",
            "address_norm"
        ],
        dtype="string"
    ).set_index("entity_id")


def calculate_training_features(
    training_pairs,
    source_name,
    source_path
):
    """
    Calculate features only for pairs belonging to one source.
    """

    pairs = training_pairs[
        training_pairs["source"] == source_name
    ].copy()

    if pairs.empty:
        return pd.DataFrame(
            index=pairs.index,
            columns=FEATURE_COLUMNS
        )

    print(
        f"Loading {source_name}: "
        f"{len(pairs):,} pairs"
    )

    source_lookup = load_source_lookup(
        source_path
    )

    print(
        f"{source_name} lookup:",
        source_lookup.shape
    )

    s1_rows = s1_lookup.reindex(
        pairs["source1_entity_id"].values
    )

    source_rows = source_lookup.reindex(
        pairs["candidate_entity_id"].values
    )

    features = []

    for s1_row, source_row in zip(
        s1_rows.itertuples(index=False),
        source_rows.itertuples(index=False)
    ):

        features.append(
            pair_features(
                s1_row.name_core,
                s1_row.address_norm,
                s1_row.country_norm,

                source_row.name_core,
                source_row.address_norm,
                source_row.country_norm
            )
        )

    features = pd.DataFrame(
        features,
        index=pairs.index
    )

    del source_lookup
    del s1_rows
    del source_rows

    gc.collect()

    return features

In [21]:
s2_training_features = calculate_training_features(
    training_pairs,
    "S2",
    S2_PATH
)

print(
    "S2 features:",
    s2_training_features.shape
)

Loading S2: 30,741 pairs
S2 lookup: (5034616, 3)
S2 features: (30741, 7)


In [22]:
s3_training_features = calculate_training_features(
    training_pairs,
    "S3",
    S3_PATH
)

print(
    "S3 features:",
    s3_training_features.shape
)

Loading S3: 9,259 pairs
S3 lookup: (5285603, 3)
S3 features: (9259, 7)


In [23]:
training_features = pd.concat(
    [
        s2_training_features,
        s3_training_features
    ]
).sort_index()

training_features = (
    training_features[FEATURE_COLUMNS]
    .fillna(0.0)
)

print(
    "Final feature matrix:",
    training_features.shape
)

display(
    training_features.head()
)

Final feature matrix: (40000, 7)


,name_char_similarity,name_3gram_jaccard,name_token_jaccard,address_char_similarity,address_token_jaccard,country_exact,translit_name_char_similarity
0,0.533333,0.238095,0.5,0.362319,0.100000,1.0,0.533333
1,1.000000,1.000000,1.0,0.861111,0.333333,1.0,1.000000
2,1.000000,1.000000,1.0,0.366667,0.000000,1.0,1.000000
3,1.000000,1.000000,1.0,0.426230,0.714286,1.0,1.000000
4,1.000000,1.000000,1.0,0.281250,0.000000,1.0,1.000000


In [24]:
print("Missing values:")
print(
    training_features.isna().sum()
)

print("\nFeature statistics:")
display(
    training_features.describe().T
)

Missing values:
name_char_similarity             0
name_3gram_jaccard               0
name_token_jaccard               0
address_char_similarity          0
address_token_jaccard            0
country_exact                    0
translit_name_char_similarity    0
dtype: int64

Feature statistics:


,count,mean,std,min,25%,50%,75%,max
name_char_similarity,40000.0,0.975210,0.088222,0.272727,1.000000,1.000000,1.000000,1.0
name_3gram_jaccard,40000.0,0.956774,0.145627,0.000000,1.000000,1.000000,1.000000,1.0
name_token_jaccard,40000.0,0.966456,0.122949,0.166667,1.000000,1.000000,1.000000,1.0
address_char_similarity,40000.0,0.546668,0.283555,0.000000,0.333333,0.433333,0.842105,1.0
address_token_jaccard,40000.0,0.340130,0.370619,0.000000,0.000000,0.111111,0.666667,1.0
country_exact,40000.0,0.996475,0.059268,0.000000,1.000000,1.000000,1.000000,1.0
translit_name_char_similarity,40000.0,0.975210,0.088222,0.272727,1.000000,1.000000,1.000000,1.0


In [25]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    fbeta_score
)

X = training_features.copy()

y = (
    training_pairs["label"]
    .astype(int)
    .reset_index(drop=True)
)

groups = (
    training_pairs["source1_entity_id"]
    .astype(str)
    .reset_index(drop=True)
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

print("Training:")
print(X_train.shape)
print(y_train.value_counts())

print("\nValidation:")
print(X_val.shape)
print(y_val.value_counts())

Training:
(31846, 7)
label
1    16074
0    15772
Name: count, dtype: int64

Validation:
(8154, 7)
label
0    4228
1    3926
Name: count, dtype: int64


In [26]:
model = LogisticRegression(
    max_iter=1000,
    C=2.0,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

print("Model trained successfully.")

Model trained successfully.


In [27]:
val_probabilities = model.predict_proba(
    X_val
)[:, 1]

roc_auc = roc_auc_score(
    y_val,
    val_probabilities
)

average_precision = average_precision_score(
    y_val,
    val_probabilities
)

print(f"ROC-AUC:          {roc_auc:.6f}")
print(f"Average Precision: {average_precision:.6f}")

ROC-AUC:          0.993786
Average Precision: 0.992464


In [28]:
thresholds = np.arange(
    0.30,
    0.991,
    0.01
)

threshold_results = []

for threshold in thresholds:

    predictions = (
        val_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        predictions,
        zero_division=0
    )

    f05 = fbeta_score(
        y_val,
        predictions,
        beta=0.5,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "F0.5": f05,
        "TP": int(
            ((predictions == 1) & (y_val == 1)).sum()
        ),
        "FP": int(
            ((predictions == 1) & (y_val == 0)).sum()
        ),
        "FN": int(
            ((predictions == 0) & (y_val == 1)).sum()
        )
    })

threshold_df = pd.DataFrame(
    threshold_results
)

display(
    threshold_df.sort_values(
        "F0.5",
        ascending=False
    ).head(15)
)

,threshold,precision,recall,F0.5,TP,FP,FN
41,0.71,0.986970,0.906775,0.969816,3560,47,366
40,0.70,0.986707,0.907539,0.969788,3563,48,363
33,0.63,0.984632,0.913907,0.969625,3588,56,338
35,0.65,0.984886,0.912888,0.969592,3584,55,342
42,0.72,0.986955,0.905756,0.969571,3556,47,370
34,0.64,0.984628,0.913653,0.969564,3587,56,339
37,0.67,0.985395,0.910851,0.969526,3576,53,350
32,0.62,0.984366,0.914162,0.969476,3589,57,337
39,0.69,0.985908,0.908813,0.969460,3568,51,358
38,0.68,0.985643,0.909322,0.969371,3570,52,356


In [29]:
best_row = threshold_df.loc[
    threshold_df["F0.5"].idxmax()
]

BEST_THRESHOLD = float(
    best_row["threshold"]
)

print(
    f"Selected threshold: {BEST_THRESHOLD:.2f}"
)

print(
    f"Validation precision: "
    f"{best_row['precision']:.6f}"
)

print(
    f"Validation recall: "
    f"{best_row['recall']:.6f}"
)

print(
    f"Validation F0.5: "
    f"{best_row['F0.5']:.6f}"
)

Selected threshold: 0.71
Validation precision: 0.986970
Validation recall: 0.906775
Validation F0.5: 0.969816


In [30]:
MODEL_ARTIFACT = {
    "model": model,
    "feature_columns": FEATURE_COLUMNS,
    "threshold": BEST_THRESHOLD,
    "roc_auc": float(roc_auc),
    "average_precision": float(average_precision)
}

joblib.dump(
    MODEL_ARTIFACT,
    MODEL_PATH
)

print("Model saved successfully:")
print(MODEL_PATH)

Model saved successfully:
/content/gdrive/MyDrive/Amazon ML Challenge 2026/04_Models/entity_resolution_logistic_model.joblib


In [31]:
print(
    "Exists:",
    os.path.exists(MODEL_PATH)
)

if os.path.exists(MODEL_PATH):
    print(
        f"Size: "
        f"{os.path.getsize(MODEL_PATH) / 1024:.2f} KB"
    )

Exists: True
Size: 1.58 KB


In [32]:
# ============================================================
# PRODUCTION SCORING CONFIGURATION
# ============================================================

CHUNK_SIZE = 250_000

SCORING_THRESHOLD = BEST_THRESHOLD

print("Scoring threshold:", SCORING_THRESHOLD)
print("Candidate file:", CANDIDATE_PATH)
print("Candidate file exists:", os.path.exists(CANDIDATE_PATH))

Scoring threshold: 0.7100000000000004
Candidate file: /content/gdrive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs/candidate_pairs.tsv
Candidate file exists: True


In [33]:
model_artifact = joblib.load(
    MODEL_PATH
)

model = model_artifact["model"]
FEATURE_COLUMNS = model_artifact["feature_columns"]
SCORING_THRESHOLD = model_artifact["threshold"]

print("Model loaded successfully.")
print()
print("Features:")
print(FEATURE_COLUMNS)
print()
print("Threshold:", SCORING_THRESHOLD)

Model loaded successfully.

Features:
['name_char_similarity', 'name_3gram_jaccard', 'name_token_jaccard', 'address_char_similarity', 'address_token_jaccard', 'country_exact', 'translit_name_char_similarity']

Threshold: 0.7100000000000004


In [34]:
def score_source_candidates(
    source_name,
    source_path,
    candidate_limit=None,
    output_path=None
):
    """
    Score candidates for one source at a time.

    candidate_limit:
        Maximum number of candidate rows to read.
        None = entire candidate file.

    Only candidates belonging to source_name are scored.
    """

    print("=" * 70)
    print(f"STARTING {source_name} SCORING")
    print("=" * 70)

    # --------------------------------------------------------
    # Load source lookup
    # --------------------------------------------------------

    print(f"\nLoading {source_name} lookup...")

    source_lookup = pd.read_csv(
        source_path,
        sep="\t",
        usecols=[
            "entity_id",
            "country_norm",
            "name_core",
            "address_norm"
        ],
        dtype="string"
    ).set_index("entity_id")

    print(
        f"{source_name} lookup loaded:",
        source_lookup.shape
    )

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    stats = {
        "source": source_name,
        "candidate_rows_read": 0,
        "source_rows": 0,
        "valid_rows": 0,
        "cheap_survivors": 0,
        "feature_rows": 0,
        "predicted_matches": 0
    }

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    if output_path is not None:

        if os.path.exists(output_path):
            os.remove(output_path)

        output_header_written = False

    # --------------------------------------------------------
    # Read candidate file
    # --------------------------------------------------------

    reader = pd.read_csv(
        CANDIDATE_PATH,
        sep="\t",
        usecols=[
            "source1_entity_id",
            "candidate_entity_id",
            "source"
        ],
        dtype="string",
        chunksize=CHUNK_SIZE
    )

    for chunk_number, chunk in enumerate(
        reader,
        start=1
    ):

        # ----------------------------------------------------
        # Candidate limit
        # ----------------------------------------------------

        if candidate_limit is not None:

            remaining = (
                candidate_limit
                -
                stats["candidate_rows_read"]
            )

            if remaining <= 0:
                break

            if len(chunk) > remaining:

                chunk = chunk.iloc[
                    :remaining
                ].copy()

        stats["candidate_rows_read"] += len(chunk)

        # ----------------------------------------------------
        # Select source
        # ----------------------------------------------------

        chunk = chunk[
            chunk["source"] == source_name
        ].copy()

        if chunk.empty:
            continue

        stats["source_rows"] += len(chunk)

        # ----------------------------------------------------
        # S1 lookup
        # ----------------------------------------------------

        s1_rows = s1_lookup.reindex(
            chunk["source1_entity_id"].values
        )

        # ----------------------------------------------------
        # Source lookup
        # ----------------------------------------------------

        src_rows = source_lookup.reindex(
            chunk["candidate_entity_id"].values
        )

        work = pd.DataFrame({
            "source1_entity_id":
                chunk["source1_entity_id"].values,

            "candidate_entity_id":
                chunk["candidate_entity_id"].values,

            "source":
                chunk["source"].values,

            "s1_name":
                s1_rows["name_core"].values,

            "s1_address":
                s1_rows["address_norm"].values,

            "s1_country":
                s1_rows["country_norm"].values,

            "src_name":
                src_rows["name_core"].values,

            "src_address":
                src_rows["address_norm"].values,

            "src_country":
                src_rows["country_norm"].values
        })

        # ----------------------------------------------------
        # Remove invalid lookups
        # ----------------------------------------------------

        valid_mask = (
            work["s1_name"].notna()
            &
            work["src_name"].notna()
        )

        work = work[
            valid_mask
        ].copy()

        stats["valid_rows"] += len(work)

        if work.empty:
            continue

        # ----------------------------------------------------
        # Cheap filtering
        # ----------------------------------------------------

        cheap_mask = cheap_prefilter(
            work
        )

        work = work[
            cheap_mask
        ].copy()

        stats["cheap_survivors"] += len(work)

        if work.empty:
            continue

        # ----------------------------------------------------
        # Expensive seven features
        # ----------------------------------------------------

        features = build_features_for_scoring_rows(
            work
        )

        stats["feature_rows"] += len(features)

        # ----------------------------------------------------
        # Predict
        # ----------------------------------------------------

        probabilities = model.predict_proba(
            features[FEATURE_COLUMNS]
        )[:, 1]

        match_mask = (
            probabilities >= SCORING_THRESHOLD
        )

        if match_mask.any():

            predictions = work.loc[
                match_mask,
                [
                    "source1_entity_id",
                    "candidate_entity_id",
                    "source"
                ]
            ].copy()

            predictions["probability"] = (
                probabilities[match_mask]
            )

            stats["predicted_matches"] += len(
                predictions
            )

            if output_path is not None:

                predictions.to_csv(
                    output_path,
                    sep="\t",
                    index=False,
                    mode="a",
                    header=not output_header_written
                )

                output_header_written = True

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if chunk_number % 2 == 0:

            print(
                f"Chunk {chunk_number:,} | "
                f"read={stats['candidate_rows_read']:,} | "
                f"{source_name}={stats['source_rows']:,} | "
                f"cheap={stats['cheap_survivors']:,} | "
                f"features={stats['feature_rows']:,} | "
                f"matches={stats['predicted_matches']:,}"
            )

        del chunk
        del s1_rows
        del src_rows
        del work
        del features
        del probabilities

        gc.collect()

    del source_lookup
    del reader

    gc.collect()

    print("\n" + "=" * 70)
    print(f"{source_name} COMPLETE")
    print("=" * 70)

    for key, value in stats.items():
        print(
            f"{key:25}: {value:,}"
            if isinstance(value, int)
            else f"{key:25}: {value}"
        )

    return stats

In [35]:
SMOKE_OUTPUT = os.path.join(
    SCORING_ROOT,
    "s2_smoke_test.tsv"
)

s2_smoke_stats = score_source_candidates(
    source_name="S2",
    source_path=S2_PATH,
    candidate_limit=1_000_000,
    output_path=SMOKE_OUTPUT
)

STARTING S2 SCORING

Loading S2 lookup...
S2 lookup loaded: (5034616, 3)


NameError: name 'cheap_prefilter' is not defined

In [ ]:
print("\nSmoke-test statistics:")
print(
    json.dumps(
        s2_smoke_stats,
        indent=2
    )
)

if os.path.exists(SMOKE_OUTPUT):

    smoke_predictions = pd.read_csv(
        SMOKE_OUTPUT,
        sep="\t"
    )

    print(
        "\nPredicted matches:",
        len(smoke_predictions)
    )

    display(
        smoke_predictions.head(20)
    )

    print("\nProbability distribution:")
    display(
        smoke_predictions["probability"]
        .describe()
    )
else:
    print(
        "No predicted matches passed the threshold."
    )